# IMU 数据预处理流水线 (preprocess_v2)

**与 `preprocess/`（v1）的差异**：输出采样率由「设备原生率（~99Hz，随设备浮动）」
改为**固定 100 Hz**，使不同设备、不同批次的数据落在同一条时间栅格上，可逐行并列比较。
路径参数、cell 结构、调用方式、输出格式均与 v1 一致。

**功能总览**

| Cell | 功能 | 说明 |
|------|------|------|
| Cell 0 | 全局参数配置 | 集中定义所有路径与参数，修改配置只需编辑本 cell |
| Cell 1 | 视频配准 | 加载 IMU 原始数据（变量 `raw`），启动交互式 GUI 与视频对齐，获得时间偏移 `t_diff` |
| Cell 2 | 数据处理与导出 | **复用 Cell 1 的 `raw` 与 `t_diff`**（或用 `T_DIFF_MANUAL` 手动指定），执行完整预处理并保存 CSV |

**依赖文件**（与本 notebook 同目录）：
- `imu_processor.py` — 数据导入与预处理核心库（**统一 100Hz、8 列**）
- `aligner.py` — IMU-视频时间配准 GUI（支持传入预加载的 `raw`）
- `imu_utils.py` — `save_processed_csv` 等辅助函数

**输出格式（8 列）**：`time, ax, ay, az, wx, wy, wz, interp_mask`
- 采样率固定 **100 Hz**；`time` 列为 0.01 s 的整数倍（扣除 `t_diff` 后）
- 信号已做 20Hz 低通滤波，**不做 EKF 重力分离**
- 后续切分（滑动窗口等）请基于**采样点数**而非秒；切分时同步切分 `time` 列
- 100 Hz 下：400 点 = 4.000 s（v2.5 模型窗长），不同设备可直接按行号并列

**典型流程**：先运行 Cell 0 配置参数 → 运行 Cell 1 完成配准（GUI 关闭后自动记录 `t_diff`）→ 运行 Cell 2 导出数据。
若已有已知的 `t_diff`，可跳过 Cell 1，在 Cell 0 中设置 `T_DIFF_MANUAL` 后直接运行 Cell 2。

**一般来说，一段视频对应2段IMU，可以选择两段IMU中同步信号更明显的来配准，配准一次，t_diff可以复用**

**v2.1 更新（2026-09-16）**
- 新增开关 `DO_REGISTRATION`：
  - `True` → 运行 Cell 1 交互式配准 GUI，得到 `t_diff`；
  - `False` → **跳过 GUI，Cell 2 直接读取视频同目录的 `t_diff.txt`**（可用 `T_DIFF_FILE` 指定其它路径）。
- **修复重复文件隐患**：旧版 Cell 2 用 `if "raw" not in globals()` 判断是否读盘，换 `IMU_DIR` 后 raw 仍是上一只鼠的数据 → 会把同一份数据写进两个输出文件。现改为按 `IMU_DIR` 缓存（`imu_utils.get_raw`）：**目录变了就自动重载**；并新增 `FORCE_RELOAD_RAW` 强制刷新、Cell 2 打印**输出指纹**便于核对。
- 新增 `%autoreload 2`：修改 `imu_processor.py` / `aligner.py` 后重跑 cell 即生效。

## Cell 0 — 全局参数配置 (Global Paths & Parameters)

In [52]:
# ============================================================
# Cell 0: 全局参数配置 (Global Paths & Parameters)
# 功能: 集中定义所有路径与参数, 供 Cell 1 / Cell 2 共用
# ============================================================

# 修改本目录下 imu_processor.py / aligner.py 后, 重跑 cell 自动生效, 无需重启内核
%load_ext autoreload
%autoreload 2

# --------------------- 路径配置 ---------------------
VIDEO_PATH  = r"\\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test2_vs_Tool1\data2.mp4"     # 视频文件路径 (Cell 1 配准用)
IMU_DIR     = r"\\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test2_vs_Tool1\13"            # IMU 原始 CSV 目录
OUTPUT_PATH = r"\\hulab.lumiani.ai\Data\WSB\Computational_Ethology\processed_Data\Train\6channel_100Hz\Test2_vs_Tool1-13.csv"      # 输出 CSV 路径 (Cell 2 保存用)

# --------------------- 视频配准开关 ---------------------
DO_REGISTRATION = False   # True : 运行 Cell 1 的配准 GUI, 得到 t_diff
                          # False: 跳过 GUI, Cell 2 直接读取 t_diff 文件里的数值
T_DIFF_FILE = None        # t_diff 文件路径; None = 自动取 视频同目录/t_diff.txt

# --------------------- Cell 1 (配准) 专用参数 ---------------------
IMU_DURATION = 120.0  # GUI 中仅显示 IMU 前 N 秒 (加速显示)
VIEW_WINDOW  = 30.0   # IMU 曲线初始视窗宽度 (s)

# --------------------- Cell 2 (处理导出) 专用参数 ---------------------
# 输出采样率固定 100 Hz (见 imu_processor.FS_OUT, 与设备原生率无关)
T_DIFF_MANUAL = None  # 手动指定 t_diff (秒); None = 用 t_diff 文件 / Cell 1 结果
APPLY_LPF     = True  # 是否低通滤波
LPF_FC        = 20.0  # 低通滤波截止频率 (Hz)

# --------------------- 重复运行 / 缓存控制 ---------------------
# notebook 会反复运行: 切换 IMU_DIR 后必须读到新目录的数据, 否则会把上一只鼠的
# 数据重复写到新输出 (这正是「一对小鼠文件完全相同」的根源)。
FORCE_RELOAD_RAW = True  # True: 每次 Cell 1/2 都强制重新读取 IMU 目录
# ----------------------------------------------------

print("配置完成:")
print(f"  VIDEO_PATH       = {VIDEO_PATH}")
print(f"  IMU_DIR          = {IMU_DIR}")
print(f"  OUTPUT_PATH      = {OUTPUT_PATH}")
print(f"  DO_REGISTRATION  = {DO_REGISTRATION}   (False -> 读取 t_diff 文件)")
print(f"  T_DIFF_MANUAL    = {T_DIFF_MANUAL}")
print(f"  FORCE_RELOAD_RAW = {FORCE_RELOAD_RAW}")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
配置完成:
  VIDEO_PATH       = \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test2_vs_Tool1\data2.mp4
  IMU_DIR          = \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test2_vs_Tool1\13
  OUTPUT_PATH      = \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\processed_Data\Train\6channel_100Hz\Test2_vs_Tool1-13.csv
  DO_REGISTRATION  = False   (False -> 读取 t_diff 文件)
  T_DIFF_MANUAL    = None
  FORCE_RELOAD_RAW = True


## Cell 1 — 视频配准 (Video-IMU Time Alignment)

In [33]:
# ============================================================
# Cell 1: 视频配准 (Video-IMU Time Alignment)
# 仅当 Cell 0 的 DO_REGISTRATION=True 时运行 (弹出 GUI 与视频对齐)
# DO_REGISTRATION=False 时本 cell 只提示跳过, Cell 2 会直接读取 t_diff 文件
# ============================================================
%matplotlib tk
from imu_utils import get_raw, t_diff_path
from aligner import IMUVideoAligner

if not DO_REGISTRATION:
    print("DO_REGISTRATION=False -> 跳过视频配准 GUI。")
    print("Cell 2 将直接读取 t_diff 文件:", t_diff_path(VIDEO_PATH, T_DIFF_FILE))
else:
    # 按 IMU_DIR 缓存读取: 目录变化会自动重新加载 (不会复用上一只鼠的旧数据)
    raw = get_raw(IMU_DIR, force=FORCE_RELOAD_RAW, verbose=True)
    print(f"raw 来源目录: {IMU_DIR}  shape={raw.shape}  首样本 ax={raw[0, 1]:.4f}")
    aligner = IMUVideoAligner(
        VIDEO_PATH,
        raw_data=raw,               # 传入预加载数据, 不重复读取磁盘
        lpf_fc=LPF_FC,
        imu_duration=IMU_DURATION,
        view_window=VIEW_WINDOW,
    )
    aligner.show()   # 弹出 GUI 窗口; 关闭窗口后代码继续往下执行


[get_raw] 强制重新读取 \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.25_social_all_paired\A4_C4_C4-c8\c8
Loaded 2 files, 185696 samples from \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.25_social_all_paired\A4_C4_C4-c8\c8
raw 来源目录: \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.25_social_all_paired\A4_C4_C4-c8\c8  shape=(185696, 8)  首样本 ax=57371.9720
Extracting first 120 s of video as 360p JPEG frames (3596 frames, one-time only)...
  300/3596 frames (8%)
  600/3596 frames (17%)
  900/3596 frames (25%)
  1200/3596 frames (33%)
  1500/3596 frames (42%)
  1800/3596 frames (50%)
  2100/3596 frames (58%)
  2400/3596 frames (67%)
  2700/3596 frames (75%)
  3000/3596 frames (83%)
  3300/3596 frames (92%)
Frame cache saved: \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.25_social_all_paired\A4_C4_C4-c8\data2_360p_frames (3596 frames)
Using pre-loaded raw IMU data (shape (185696, 8), first 120 s shown).


In [34]:
# 记录配准结果 (GUI 中 "Save t_diff" 同时会写入视频旁的 t_diff.txt)
if DO_REGISTRATION:
    t_diff = aligner.offset
    print(f"\n配准结果: t_diff = {t_diff:.6f} s  (Cell 2 将默认使用该值)")
else:
    print("未运行配准 (DO_REGISTRATION=False); t_diff 将在 Cell 2 由 t_diff 文件决定。")



配准结果: t_diff = 2.992095 s  (Cell 2 将默认使用该值)


## Cell 2 — 数据处理与导出 (Data Processing & Export)

In [53]:
# ============================================================
# Cell 2: 数据处理与导出 (Data Processing & Export)
# 功能: 取得 raw 与 t_diff -> 完整预处理 -> 保存 CSV
# 输出: 100 Hz, 8 列 CSV (time, ax, ay, az, wx, wy, wz, interp_mask)
# ============================================================

from imu_processor import data_preprocess
from imu_utils import get_raw, load_t_diff, save_processed_csv, t_diff_path

# ---- 1) 解析 t_diff ----
if T_DIFF_MANUAL is not None:
    t_diff = float(T_DIFF_MANUAL)
    _src = "Cell 0 的 T_DIFF_MANUAL"
elif not DO_REGISTRATION:
    t_diff = load_t_diff(VIDEO_PATH, T_DIFF_FILE)      # 直接读 t_diff 文件
    _src = f"t_diff 文件 {t_diff_path(VIDEO_PATH, T_DIFF_FILE)}"
elif "t_diff" in globals():
    _src = "Cell 1 配准结果"
else:
    raise RuntimeError("未检测到 t_diff: 请运行 Cell 1 完成配准, 或将 "
                       "DO_REGISTRATION 设为 False 以读取 t_diff 文件, "
                       "或在 Cell 0 设置 T_DIFF_MANUAL")
print(f"t_diff = {t_diff:.6f} s  (来源: {_src})")

# ---- 2) 读取 raw (按 IMU_DIR 缓存; 目录变化自动重载) ----
raw = get_raw(IMU_DIR, force=FORCE_RELOAD_RAW, verbose=True)

# ---- 3) 完整预处理 (复用 raw, 不重复读取) ----
# 输出 100 Hz, 8 列: time, ax, ay, az, wx, wy, wz, interp_mask
proc, info = data_preprocess(
    raw,
    t_diff=t_diff,
    apply_lpf=APPLY_LPF,
    lpf_fc=LPF_FC,
    return_drift=True,
)
print(f"proc shape: {proc.shape}")
print(f"fs_out = {info['fs']:.2f} Hz (device native {info['fs_native']:.2f} Hz, "
      f"drift = {info['drift']:.3f}%)")
print(f"dropped = {info['dropped_samples']}, "
      f"interpolated = {info['interpolated_samples']}")

# ---- 4) 保存 CSV (8 通道: time, ax, ay, az, wx, wy, wz, interp_mask) ----
save_processed_csv(proc, OUTPUT_PATH)

# ---- 5) 输出指纹: 立即核对本次输出确实来自当前 IMU_DIR (防止重复文件) ----
print(f"[指纹] IMU_DIR        = {IMU_DIR}")
print(f"[指纹] rows           = {proc.shape[0]}")
print(f"[指纹] mean(ax,ay,az) = ({proc[:, 1].mean():+.4f}, "
      f"{proc[:, 2].mean():+.4f}, {proc[:, 3].mean():+.4f})")
print(f"[指纹] first(ax,ay,az)= ({proc[0, 1]:+.4f}, "
      f"{proc[0, 2]:+.4f}, {proc[0, 3]:+.4f})")


t_diff = -5.488973 s  (来源: t_diff 文件 \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test2_vs_Tool1\t_diff.txt)
[get_raw] 强制重新读取 \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test2_vs_Tool1\13
Loaded 2 files, 177164 samples from \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test2_vs_Tool1\13
proc shape: (178431, 8)
fs_out = 100.00 Hz (device native 99.29 Hz, drift = 0.710%)
dropped = 0, interpolated = 3
Saved 178431 samples x 8 channels -> \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\processed_Data\Train\6channel_100Hz\Test2_vs_Tool1-13.csv
[指纹] IMU_DIR        = \\hulab.lumiani.ai\Data\WSB\Computational_Ethology\Data\8.6_2mouse_social\Test2_vs_Tool1\13
[指纹] rows           = 178431
[指纹] mean(ax,ay,az) = (-0.2309, -0.0058, +0.6732)
[指纹] first(ax,ay,az)= (-0.9569, +0.1809, +0.5500)
